In [6]:
# ====================================
# Import
# ====================================
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt

import joblib

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

# ====================================
# Definitions
# ====================================

def evaluate_pr(model, X_test, y_test):

    prob = model.predict_proba(X_test)[:, 1]

    precision, recall, thresholds = precision_recall_curve(
        y_test,
        prob
    )

    ap = average_precision_score(
        y_test,
        prob
    )

    print(ap)

    return precision, recall, ap

def plot_pr(baseline, precision, recall, ap):

    plt.plot(
        recall,
        precision,
        label=f"AP = {ap:.2f}"
    )

    plt.axhline(
        y = baseline,
        linestyle="--",
        label="Random baseline"
    )

    plt.xlabel(
        "Recall"
    )

    plt.ylabel(
        "Precision"
    )

    plt.title(
        "Precision-Recall Curve"
    )

    plt.legend()
    plt.show()


# ====================================
# Load
# ====================================

BASE_DIR = Path.cwd()

ini_file = BASE_DIR / "data" / "employees.csv"

df = pd.read_csv(ini_file)

# ====================================
# Data Cleaning
# ====================================

df["PerformanceScore"] = df["PerformanceScore"].fillna(
    df["PerformanceScore"].mode()[0]
)
df["Education"] = df["Education"].fillna(
    df["Education"].mode()[0]
)

# ====================================
# Creating unbalanced data
# ====================================

df["HighSalary"] = (
    df["Salary"] >= df["Salary"].median()
)

df_false = df[
    df["HighSalary"] == False
]

df_true = df[
    df["HighSalary"] == True
]

df_true = df_true.sample(
    n=10,
    random_state=42
)

df = pd.concat(
    [
        df_false,
        df_true
    ]
)

# ====================================
# Feature Selection
# ====================================

X = df[[
    "PerformanceScore",
    "Education",
    "Department",
    "Experience"
]].copy()


# ====================================
# Feature Engineering
# ====================================

X["ExperienceScore"] = (
    X["Experience"] * X["PerformanceScore"]
)

# ====================================
# Target
# ====================================

y = df["HighSalary"]

# ====================================
# Train/Test Split
# ====================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ====================================
# Feature split
# ====================================

numeric_features = [
    "PerformanceScore",
    "Experience",
    "ExperienceScore"
]

categorical_features = [
    "Education",
    "Department"
]

# ====================================
# ColumnTransformer
# ====================================

preprocessor = ColumnTransformer([
    (
        "numeric",
        StandardScaler(),
        numeric_features
    ),
    (
        "categorical",
        OneHotEncoder(drop="first"), 
        categorical_features
    )
])

# ====================================
# Models
# ====================================

logistic_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        LogisticRegression(
            class_weight="balanced"
        )
    )
])

forest_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        RandomForestClassifier(
            max_depth=5,
            n_estimators=100,
            random_state=42
        )
    )
])

svm_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        SVC(
            kernel="linear",
            probability=True
        )
    )
])



# ====================================
# Gradient Model
# ====================================

gradient_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        GradientBoostingClassifier(
            random_state=42
        )
    )
])


# Models

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": forest_model,
    "SVM": svm_model,
    "Gradient Boosting": gradient_model
}

# Fit

results = []

for name, model in models.items():

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_test
    )

    prob = model.predict_proba(X_test)[:, 1]

    ap = average_precision_score(
        y_test,
        prob
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(
            y_test,
            pred
        ),
        "Precision": precision_score(
            y_test,
            pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            pred,
            zero_division=0
        ),
        "Average Precision": ap
    })

results_df = pd.DataFrame(
    results
)

print(results_df)
# ====================================
# Model saving
# ====================================

model_path = BASE_DIR / "models" / "gradient_model.pkl"

# joblib.dump(
#     gradient_model,
#     model_path
# )


                 Model  Accuracy  Precision  Recall        F1  \
0  Logistic Regression  0.642857   0.166667     1.0  0.285714   
1        Random Forest  0.928571   0.000000     0.0  0.000000   
2                  SVM  0.928571   0.000000     0.0  0.000000   
3    Gradient Boosting  0.785714   0.000000     0.0  0.000000   

   Average Precision  
0           0.500000  
1           0.333333  
2           0.250000  
3           0.166667  


c:\Users\Drtic123\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
